# Welfare gating on the Balanced-hybrid RS: threshold sweep and paradigm heterogeneity

This notebook redoes the threshold analysis of Uğurel & Yabe (2026) on the current engine: all 8 metros,
2,000 agents, 60 days, 5 seeds. Every gated arm is the **Balanced hybrid** recommender with a welfare gate
on top. The gate takes each candidate and either passes it or withholds it:

* **PUP** (positive-utility probability) passes a venue only if $\Pr[U \ge 0] \ge \alpha$.
* **RM** (regret minimisation) passes a venue only if $\mathbb{E}[\text{regret}] \le \varepsilon$.

When every candidate is withheld, the RS *abstains* and the agent goes wherever it would have gone organically.

**Plan**
1. Run the two controls (No RS and Oracle) and the ungated Balanced hybrid (Standard RS).
2. Calibrate the RM grid on percentiles of the expected-regret distribution under Standard RS, as the paper does.
3. Sweep PUP $\alpha$ and RM $\varepsilon$, then pick each optimum (highest mean net utility, paper Figs 2–3).
4. Run PUP+RM at both optima and break the welfare change down by decision paradigm (paper Fig 4).

**Metric.** As in the paper (Table 1): the mean net utility $U = V - C$ of leisure trips on the last (evaluation)
day. Each (city, seed) run gives one mean. Cities are averaged with equal weight, so each seed yields one number.
Error bars are the standard deviation of that number across the 5 seeds ($\sigma_U$).

Each finished run is cached under `.cache/gating_sweep/`. An interrupted notebook picks up where it
stopped, and a full re-run only redraws the figures.

## 1. Setup

In [ ]:
import math
import multiprocessing as mp
import os
import pickle
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

# The server datasource (Postgres: POIs, ACS/PUMS populations, survey personas) is configured in .env.server.
for line in Path(".env.server").read_text().splitlines():
    if line.strip() and not line.lstrip().startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"'))

from welfare_rs import TravelCostEstimator, WelfareAwareOrchestrator, params
from welfare_rs.datasource import get_datasource
from welfare_rs.experiment_harness import gini_coefficient, leisure_net_utility, table1_metrics
from welfare_rs.metro import build_network
from welfare_rs.oracle_rs import oracle_factory
from welfare_rs.recommender_systems import RECOMMENDER_PRESETS, RecommenderConfig
from welfare_rs.simulation import Simulation

In [ ]:
# Experiment size
CITIES = ["houston", "la", "chicago", "sf_bay", "nyc", "dc", "seattle", "miami"]  # largest network first
N_AGENTS = 2000
N_DAYS = 60
SEEDS = [0, 1, 2, 3, 4]

# Threshold grids. PUP alpha = 0 passes every candidate, so it is identical to Standard RS and is not re-run.
PUP_ALPHAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
RM_PERCENTILES = [50, 60, 70, 80, 90, 95]

# One run holds 2-5 GB of private memory; the routing matrices are shared between runs through the page cache.
N_WORKERS = 40

BALANCED_HYBRID = RecommenderConfig(label="Balanced hybrid", **RECOMMENDER_PRESETS["Balanced hybrid"])
PARADIGMS = ["utility", "regret", "prospect", "satisficing", "habit"]  # the paper's row order

CACHE_DIR = Path(".cache/gating_sweep") / f"{N_AGENTS}agents_{N_DAYS}days"
FIG_DIR = Path("figs")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# Figure style (reference data-viz palette)
BLUE, ORANGE, AQUA, GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8984"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
})

print(f"{len(CITIES)} cities x {len(SEEDS)} seeds, {N_AGENTS} agents, {N_DAYS} days")

### Arms and the single-run function

An `Arm` is one recommender condition. Gated arms wrap the Balanced hybrid in the engine's
`WelfareAwareOrchestrator`, wired exactly as the web app wires a gated recommender (`backend/service.py`).

Standard RS goes through the same wrapper in pass-through mode (`"standard"`), which records the expected regret of every
candidate it is shown. Those values calibrate the RM grid. The pass-through filters nothing and draws no
random numbers, so the run is identical to an unwrapped Balanced hybrid.

In [ ]:
@dataclass(frozen=True)
class Arm:
    gate: str                   # "no_rs", "oracle", "standard", "pup", "rm" or "pup_rm"
    alpha: float = 0.0          # PUP: pass a venue only if P(U >= 0) >= alpha
    epsilon: float = math.inf   # RM:  pass a venue only if E[regret] <= epsilon

    @property
    def key(self):
        parts = [self.gate]
        if self.gate in ("pup", "pup_rm"):
            parts.append(f"a{self.alpha:.2f}")
        if self.gate in ("rm", "pup_rm"):
            parts.append(f"e{self.epsilon:.4f}")
        return "_".join(parts)


class RegretLoggingGate(WelfareAwareOrchestrator):
    # The engine's welfare gate, also recording E[regret] of every candidate it passes.
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.regrets = []

    def recommend(self, user, leisure_subtype, top_k_per_system=5):
        passed = super().recommend(user, leisure_subtype, top_k_per_system)
        self.regrets.extend(rec.expected_regret for recs in passed.values() for rec in recs)
        return passed


def recommender_factory(arm):
    if arm.gate == "no_rs":
        return None
    if arm.gate == "oracle":
        return oracle_factory("Oracle")
    mode = "standard" if arm.gate == "standard" else arm.gate

    def factory(_sim, base_stack):
        return RegretLoggingGate(base_stack, TravelCostEstimator(), mode=mode,
                                 pup_alpha=arm.alpha, rm_epsilon=arm.epsilon)
    return factory


@lru_cache(maxsize=1)  # a worker keeps the network of the city it last ran
def load_network(city):
    return build_network(city)


def run_one(city, seed, arm):
    # One simulation. Population and network setup mirror the web app's study runner.
    ds = get_datasource()
    population = ds.population(city, replicate=seed % ds.population_replicates(city),
                               limit=N_AGENTS + max(100, N_AGENTS // 5))
    if population is None or len(population) < N_AGENTS:
        raise RuntimeError(f"{city}: no built population with {N_AGENTS} agents")
    sim = Simulation(
        num_agents=N_AGENTS, seed=seed,
        use_recommenders=(arm.gate != "no_rs"),
        persona_rows=ds.personas(),
        persona_csv_path=ds.persona_csv_path() or params.SURVEY_PERSONA_CSV_PATH,
        population_rows=population,
        road_network=load_network(city),
        poi_rows=ds.poi_rows(city),
        disabled_modes=("transit",),
        recommender_config=BALANCED_HYBRID,
        recommender_factory=recommender_factory(arm),
    )
    sim.run_days(N_DAYS)
    # After run_days, agent.trips holds the evaluation (last) day.
    trips = [(agent.decision_paradigms["mode_choice"], leisure_net_utility(trip))
             for agent in sim.agents for trip in agent.trips if trip.purpose == "leisure"]
    return {
        "city": city, "seed": seed, "arm": arm.key,
        **table1_metrics(sim),
        "trips": trips,
        "regrets": np.asarray(sim.recommender_stack.regrets, dtype=np.float32) if arm.gate == "standard" else None,
    }

### Parallel runner

Each `(city, seed, arm)` run is one task on a process pool. Results are written to disk as they finish. A failed run
is reported and does not stop the others; re-running the cell retries only the runs that are missing.

In [ ]:
def cache_path(city, seed, arm):
    return CACHE_DIR / f"{city}_seed{seed}_{arm.key}.pkl"


def run_grid(arms):
    # Run every (city, seed, arm) not yet cached. Returns one row per run.
    tasks = [(city, seed, arm) for arm in arms for city in CITIES for seed in SEEDS]
    todo = [task for task in tasks if not cache_path(*task).exists()]
    # Oracle (the slowest arm) and big cities first, so the grid does not end on a long straggler.
    todo.sort(key=lambda task: (task[2].gate != "oracle", CITIES.index(task[0])))
    print(f"{len(tasks) - len(todo)} of {len(tasks)} runs cached; running {len(todo)} on {N_WORKERS} workers")

    failed, start = [], time.time()
    if todo:
        # fork: the workers inherit the functions defined in this notebook.
        with ProcessPoolExecutor(N_WORKERS, mp_context=mp.get_context("fork")) as pool:
            futures = {pool.submit(run_one, *task): task for task in todo}
            for done, future in enumerate(as_completed(futures), 1):
                city, seed, arm = futures[future]
                elapsed = f"{(time.time() - start) / 3600:.1f} h"
                try:
                    result = future.result()
                except Exception as exc:
                    failed.append(futures[future])
                    print(f"[{done}/{len(todo)}] {city} seed {seed} {arm.key}  FAILED: {exc!r}  ({elapsed})")
                    continue
                path = cache_path(city, seed, arm)
                tmp = path.with_suffix(".tmp")
                tmp.write_bytes(pickle.dumps(result))
                tmp.replace(path)  # atomic: an interrupted write never leaves a half-written cache file
                print(f"[{done}/{len(todo)}] {city} seed {seed} {arm.key}  U = {result['mean_utility']:+.3f}  ({elapsed})")
    if failed:
        raise RuntimeError(f"{len(failed)} runs failed; re-run this cell to retry them")
    return pd.DataFrame([pickle.loads(cache_path(*task).read_bytes()) for task in tasks])


def seed_stats(runs, column):
    # Mean of `column` over cities (equal weight) per (arm, seed), then mean and std over seeds.
    per_seed = runs.groupby(["arm", "seed"])[column].mean()
    return per_seed.groupby(level="arm").agg(["mean", lambda x: x.std(ddof=0)]).set_axis(["mean", "std"], axis=1)

## 2. Controls and Standard RS

* **No RS**: agents choose venues organically. Every welfare change is measured against this baseline.
* **Oracle**: offers the venue with the highest true net utility, so it bounds what any recommender can achieve.
* **Standard RS**: the Balanced hybrid with no gate.

In [ ]:
NO_RS, ORACLE, STANDARD = Arm("no_rs"), Arm("oracle"), Arm("standard")
base_runs = run_grid([NO_RS, ORACLE, STANDARD])

base = seed_stats(base_runs, "mean_utility")
REFS = {name: base.loc[arm.key, "mean"] for name, arm in
        [("No RS", NO_RS), ("Standard RS", STANDARD), ("Oracle", ORACLE)]}
base.rename(index={NO_RS.key: "No RS", ORACLE.key: "Oracle", STANDARD.key: "Standard RS"}).round(4)

## 3. Calibrate the RM grid

RM thresholds have no natural scale, so the paper sets $\varepsilon$ at percentiles of the expected regret that
Standard RS's recommendations carry. The percentiles are taken over every candidate recommended in every city, seed and day,
which gives one $\varepsilon$ grid shared by all cities. A lower percentile is a stricter gate: at $p_{50}$, roughly half of Standard RS's
recommendations would be withheld.

In [ ]:
regrets = np.concatenate(base_runs.loc[base_runs["arm"] == STANDARD.key, "regrets"].to_list())
RM_EPSILONS = {p: round(float(np.percentile(regrets, p)), 4) for p in RM_PERCENTILES}

print(f"{len(regrets):,} Standard-RS recommendations")
pd.Series(RM_EPSILONS, name="epsilon").rename_axis("percentile").to_frame()

## 4. Sweep the thresholds

Both grids run in one pool: 10 PUP and 6 RM thresholds, 640 runs in all.

In [ ]:
PUP_ARMS = {alpha: Arm("pup", alpha=alpha) for alpha in PUP_ALPHAS}
RM_ARMS = {p: Arm("rm", epsilon=eps) for p, eps in RM_EPSILONS.items()}
sweep_runs = run_grid([*PUP_ARMS.values(), *RM_ARMS.values()])

utility = seed_stats(pd.concat([base_runs, sweep_runs]), "mean_utility")
abstention = seed_stats(pd.concat([base_runs, sweep_runs]), "abstention_rate")

In [ ]:
def plot_frontier(x, arm_keys, xlabel, best_index, point_labels=None):
    # Paper-style threshold frontier: mean net utility (top) and abstention rate (bottom) on a shared threshold axis.
    fig, (ax_u, ax_a) = plt.subplots(2, 1, sharex=True, figsize=(8.5, 6.8), height_ratios=[3, 2])
    u, a = utility.loc[arm_keys], abstention.loc[arm_keys]

    for (name, value), color in zip(REFS.items(), [GRAY, ORANGE, AQUA]):
        ax_u.axhline(value, color=color, linestyle="--", linewidth=1.5, label=f"{name} ({value:.3f})")
    ax_u.errorbar(x, u["mean"], yerr=u["std"], color=BLUE, marker="o", markersize=6, linewidth=2,
                  capsize=3, label="Gated Balanced hybrid")
    ax_u.plot(x[best_index], u["mean"].iloc[best_index], "o", markersize=15, markerfacecolor="none",
              markeredgecolor=BLUE, markeredgewidth=1.5, label="Optimum")
    if point_labels:
        for xi, yi, text in zip(x, u["mean"], point_labels):
            ax_u.annotate(text, (xi, yi), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color="#52514e")
    ax_u.set_ylabel("Mean net utility\nper leisure trip")
    ax_u.legend(loc="best", fontsize=9, frameon=False)

    ax_a.errorbar(x, a["mean"], yerr=a["std"], color=BLUE, marker="s", markersize=6, linewidth=2, capsize=3)
    ax_a.set_ylim(0, 1.05)
    ax_a.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax_a.set_ylabel("Abstention rate")
    ax_a.set_xlabel(xlabel)
    fig.align_ylabels()
    fig.tight_layout()
    return fig

## 5. PUP threshold frontier (paper Fig. 2)

$\alpha = 0$ is Standard RS. Raising $\alpha$ withholds recommendations that are less likely to leave the agent better off.
The optimum $\alpha^*$ is the threshold with the highest mean net utility.

In [ ]:
alphas = [0.0, *PUP_ALPHAS]
pup_keys = [STANDARD.key, *(arm.key for arm in PUP_ARMS.values())]
best = int(np.argmax(utility.loc[pup_keys, "mean"]))
ALPHA_STAR = alphas[best]

fig = plot_frontier(alphas, pup_keys, r"PUP threshold $\alpha$", best)
fig.axes[1].set_xticks(alphas)
fig.savefig(FIG_DIR / "gating_pup_frontier.png", bbox_inches="tight")
plt.show()

table = pd.DataFrame({"alpha": alphas, "mean_utility": utility.loc[pup_keys, "mean"].values,
                      "sigma_u": utility.loc[pup_keys, "std"].values,
                      "abstention": abstention.loc[pup_keys, "mean"].values})
print(f"alpha* = {ALPHA_STAR:.1f}")
table.round(4)

## 6. RM threshold frontier (paper Fig. 3)

Each point is one calibrated $\varepsilon$, labelled with its percentile. A lower $\varepsilon$ tolerates less expected regret and withholds more.

In [ ]:
rm_keys = [arm.key for arm in RM_ARMS.values()]
epsilons = list(RM_EPSILONS.values())
best = int(np.argmax(utility.loc[rm_keys, "mean"]))
P_STAR, EPSILON_STAR = RM_PERCENTILES[best], epsilons[best]

fig = plot_frontier(epsilons, rm_keys, r"RM regret ceiling $\varepsilon$", best,
                    point_labels=[f"p{p}" for p in RM_PERCENTILES])
fig.savefig(FIG_DIR / "gating_rm_frontier.png", bbox_inches="tight")
plt.show()

table = pd.DataFrame({"percentile": RM_PERCENTILES, "epsilon": epsilons,
                      "mean_utility": utility.loc[rm_keys, "mean"].values,
                      "sigma_u": utility.loc[rm_keys, "std"].values,
                      "abstention": abstention.loc[rm_keys, "mean"].values})
print(f"epsilon* = {EPSILON_STAR:.4f} (p{P_STAR})")
table.round(4)

## 7. Heterogeneity by decision paradigm (paper Fig. 4)

Each cell is the change in mean evaluation-day leisure utility relative to No RS for one decision paradigm. The change is computed
within each (city, seed), where the agents are identical across arms, and then averaged over seeds and cities.
The final arm, PUP+RM, applies both gates at their optima.

In [ ]:
PUP_STAR = Arm("pup", alpha=ALPHA_STAR) if ALPHA_STAR > 0 else STANDARD
RM_STAR = RM_ARMS[P_STAR]
PUP_RM_STAR = Arm("pup_rm", alpha=ALPHA_STAR, epsilon=EPSILON_STAR)
combined_runs = run_grid([PUP_RM_STAR])
runs = pd.concat([base_runs, sweep_runs, combined_runs], ignore_index=True)

CONDITIONS = {  # column label -> arm
    "Standard RS": STANDARD,
    f"PUP ($\\alpha$={ALPHA_STAR:.1f})": PUP_STAR,
    f"RM ($\\varepsilon$={EPSILON_STAR:.3f})": RM_STAR,
    "PUP+RM": PUP_RM_STAR,
    "Oracle": ORACLE,
}

In [ ]:
# Mean trip utility per (arm, city, seed, paradigm)
trips = pd.DataFrame([(run.arm, run.city, run.seed, paradigm, u)
                      for run in runs.itertuples() for paradigm, u in run.trips],
                     columns=["arm", "city", "seed", "paradigm", "utility"])
cell_means = trips.groupby(["arm", "city", "seed", "paradigm"])["utility"].mean()

# Change vs No RS within each (city, seed, paradigm), then averaged over cities and seeds
delta = pd.DataFrame({
    label: (cell_means.loc[arm.key] - cell_means.loc[NO_RS.key]).groupby(level="paradigm").mean()
    for label, arm in CONDITIONS.items()
}).loc[PARADIGMS]

fig, ax = plt.subplots(figsize=(9, 4.8))
vmax = np.abs(delta.values).max()
im = ax.imshow(delta.values, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
for (i, j), value in np.ndenumerate(delta.values):
    ax.text(j, i, f"{value:+.3f}", ha="center", va="center", fontsize=11,
            color="white" if abs(value) > 0.6 * vmax else "#0b0b0b")
ax.set_xticks(range(delta.shape[1]), delta.columns, rotation=15, ha="right")
ax.set_yticks(range(delta.shape[0]), [p.capitalize() for p in delta.index])
ax.set_xlabel("Condition")
ax.set_ylabel("Agent decision paradigm")
ax.grid(False)
for side in ("top", "right"):
    ax.spines[side].set_visible(True)
fig.colorbar(im, ax=ax, shrink=0.85, label=r"$\Delta$ mean net utility vs No RS")
fig.tight_layout()
fig.savefig(FIG_DIR / "gating_paradigm_heatmap.png", bbox_inches="tight")
plt.show()

delta.round(4)

### Summary of the selected conditions (paper Table 1)

Each run's Gini and negative-utility rate are computed over its evaluation-day trip utilities and then averaged like the mean.
No RS shows 100% abstention by definition, since it never recommends.

In [ ]:
stats = {column: seed_stats(runs, column) for column in ["mean_utility", "neg_rate", "abstention_rate", "gini"]}
summary = pd.DataFrame({
    label: {
        "mean_utility": stats["mean_utility"].loc[arm.key, "mean"],
        "sigma_u": stats["mean_utility"].loc[arm.key, "std"],
        "neg_rate": stats["neg_rate"].loc[arm.key, "mean"],
        "abstention": stats["abstention_rate"].loc[arm.key, "mean"],
        "gini": stats["gini"].loc[arm.key, "mean"],
    }
    for label, arm in {"No RS": NO_RS, **CONDITIONS}.items()
}).T
summary.round(4)